# Annotate temple/devotional images with Qwen2.5-VL (self-hosted, free)

Same schema and prompt as `Claude_Image_Annotation.ipynb`, but running a self-hosted
open-source vision-language model instead of a paid per-image API call. Trade-off:
no per-image cost, but you need a GPU runtime and the model may be less reliable than
Claude on this strict structured-extraction/no-hallucination schema -- worth spot-checking
a handful of images against both to see if the quality gap actually matters for your use.

**Requires a GPU runtime**: Runtime -> Change runtime type -> T4 GPU (or better). The 7B
model needs ~15GB+ VRAM in bf16 -- if it OOMs on a T4, drop to `Qwen/Qwen2.5-VL-3B-Instruct`
(smaller, faster, somewhat less reliable) rather than fighting for more VRAM.

In [ ]:
!pip install -q "transformers>=4.49.0" accelerate qwen-vl-utils bitsandbytes

In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

# Confirmed to exist with this exact id/class as of this notebook's writing. If you want to
# try Qwen3-VL instead, confirm its exact HF repo id + transformers class name first --
# don't guess, a wrong class name here fails at import, not at a helpful error message.
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# 4-bit quantization -- fits comfortably in a T4's 16GB alongside the vision encoder and
# KV cache, and buys enough headroom to use the more capable 7B instead of dropping to 3B.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=quant_config, device_map="auto"
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print("Loaded", MODEL_ID, "in 4-bit")

## 1. Upload or point at your images
For a handful: use the file picker below. For many: zip them locally, upload the zip, then
`!unzip -q yourfile.zip -d /content/images`. Or mount Drive and point `IMAGE_DIR` there.

In [ ]:
# EDIT ME
IMAGE_DIR = "/content/images"

import os
os.makedirs(IMAGE_DIR, exist_ok=True)

from google.colab import files
uploaded = files.upload()  # picks files directly into the Colab VM's working directory
for fname in uploaded:
    os.replace(fname, os.path.join(IMAGE_DIR, fname))

image_paths = [
    os.path.join(IMAGE_DIR, f) for f in sorted(os.listdir(IMAGE_DIR))
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
]
print(f"{len(image_paths)} images in {IMAGE_DIR}")

## 2. Same annotation prompt as the Claude notebook

In [ ]:
ANNOTATION_PROMPT = """You are an expert visual annotator building a dataset of Indian temples.

Analyze the provided image carefully.

Extract ONLY information that can be supported by visible evidence.

Do NOT guess or hallucinate:
- temple name
- deity name
- location
- historical facts
- mythology
- festival name
- religious tradition

If something cannot be determined from the image, use null or an empty list.

Return ONLY valid JSON using this exact schema:

{
  "image_description": "",
  "temple_name": null,
  "deity": null,
  "architecture": {
    "style": [],
    "materials": [],
    "roof": [],
    "tower": [],
    "features": []
  },
  "structures": [],
  "sculptures": [],
  "religious_symbols": [],
  "rituals": [],
  "festival_activity": [],
  "people": [],
  "environment": [],
  "visible_text": [],
  "colors": [],
  "objects": [],
  "visual_tags": [],
  "short_caption": "",
  "detailed_caption": "",
  "confidence": 0.0
}

For every observation, prioritize visible evidence.

Architecture:
Identify visible characteristics such as:
- gopuram
- vimana
- mandapa
- pillars
- stone construction
- wooden construction
- tiled roof
- carvings
- arches
- entrance structures
- walls
- courtyard

Structures:
List important visible temple structures.

Sculptures:
Describe visible statues, carvings, animals, motifs and decorative elements.

Religious symbols:
Describe only symbols that are actually visible.

Rituals and festival activity:
Describe what people are visibly doing.
Do not identify a festival unless there is clear visual evidence.

Visible text:
Transcribe readable text exactly as seen.
Do not correct or invent text.

Visual tags:
Generate 5–15 concise tags describing the visible content.
Do not use guessed temple names or locations.

short_caption:
One accurate sentence.

detailed_caption:
A detailed 30–80 word description of the image.

confidence:
Overall confidence from 0.0 to 1.0.

Return JSON only."""

In [ ]:
import json
import re

def _extract_json(text):
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            return json.loads(match.group(0))
        raise

@torch.no_grad()
def annotate_image(path, max_new_tokens=1200):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": path},
            {"type": "text", "text": ANNOTATION_PROMPT},
        ],
    }]
    text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated = model.generate(**inputs, max_new_tokens=max_new_tokens)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated)]
    raw = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    try:
        return _extract_json(raw)
    except json.JSONDecodeError:
        print(f"  unparseable JSON for {path}: {raw[:200]!r}")
        return None

SAMPLE_N = 5  # raise to len(image_paths) once output looks right
OUTPUT_PATH = "annotations_qwenvl.jsonl"

# Resume support: skip files already annotated in a prior run of this cell (protects
# a few-thousand-image job against losing completed work to an idle disconnect).
already_done = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f:
        for line in f:
            already_done.add(json.loads(line)["file"])
    print(f"{len(already_done)} images already annotated, skipping those")

remaining = [p for p in image_paths if os.path.basename(p) not in already_done]

with open(OUTPUT_PATH, "a") as out_f:
    for path in remaining[:SAMPLE_N]:
        result = annotate_image(path)
        if result is None:
            print(f"skipped (unparseable): {path}")
            continue
        record = {"file": os.path.basename(path), **result}
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()
        print(f"{os.path.basename(path)}: {result.get('short_caption')} (confidence={result.get('confidence')})")

print(f"\nSaved to {OUTPUT_PATH}")

In [ ]:
SAMPLE_N = 5
OUTPUT_PATH = "annotations_qwenvl.jsonl"

with open(OUTPUT_PATH, "a") as out_f:
    for path in image_paths[:SAMPLE_N]:
        result = annotate_image(path)
        if result is None:
            print(f"skipped (unparseable): {path}")
            continue
        record = {"file": os.path.basename(path), **result}
        out_f.write(json.dumps(record) + "\n")
        out_f.flush()
        print(f"{os.path.basename(path)}: {result.get('short_caption')} (confidence={result.get('confidence')})")

print(f"\nSaved to {OUTPUT_PATH}")

Compare a few rows here against `annotations.jsonl` from the Claude notebook on the *same*
images, if you ran both -- that tells you whether the free self-hosted route is good enough
or whether the paid API's accuracy on this hallucination-sensitive schema is worth keeping
for the images that matter most. Once you're happy with either, both feed
`BLIP_Devotional_Finetune.ipynb` the same way via `short_caption`/`detailed_caption`.